In [ ]:
import pandas as pd
from pathlib import Path
import sys

"""
Script to process logs.
It performs the following steps:
1. Finds all folders starting with 'S' in the base path.
2. In each of them, recursively searches for 'predictions_log.csv'.
3. EXTRACTS 'myo_inf_time' and CALCULATES 'processing_time_ms' (diff from 'timestamp').
4. Concatenates data into separate DataFrames.
5. Calculates statistics for both.
6. Saves aggregated data and statistics in separate folders:
   - ../data/all/myo_inf/
   - ../data/all/proc_time/
"""

def process_thesis_logs(base_repo_path):
    base_path = Path(base_repo_path)
    
    # 1. Path Definitions
    script_dir = Path.cwd().resolve()
    
    # Define OUTPUT path for MYO_INF_TIME
    myo_inf_output_dir = script_dir.parent / "data" / "all" / "myo_inf"
    
    # NEW: Define OUTPUT path for PROCESSING_TIME
    proc_time_output_dir = script_dir.parent / "data" / "all" / "proc_time"
    
    # Create output directories if they don't exist
    try:
        myo_inf_output_dir.mkdir(parents=True, exist_ok=True)
        proc_time_output_dir.mkdir(parents=True, exist_ok=True)
        print(f"Output folder 'myo_inf' created (or already exists): {myo_inf_output_dir}")
        print(f"Output folder 'proc_time' created (or already exists): {proc_time_output_dir}")
    except Exception as e:
        print(f"Fatal Error: Cannot create output folders.")
        print(f"Details: {e}")
        sys.exit(1) # Exit the script

    # Output files for MYO_INF_TIME
    myo_inf_csv_file = myo_inf_output_dir / "compiled_myo_inf_time.csv"
    myo_inf_stats_file = myo_inf_output_dir / "statistics_myo_inf.txt"

    # NEW: Output files for PROCESSING_TIME
    proc_time_csv_file = proc_time_output_dir / "compiled_processing_time_ms.csv"
    proc_time_stats_file = proc_time_output_dir / "statistics_proc_time.txt"

    # Lists to hold all data series
    all_myo_data_series = []
    all_proc_time_series = [] # NEW
    
    print(f"\nStarting search in: {base_path}")

    # 2. Find all folders starting with 'S'
    s_folders = [f for f in base_path.iterdir() if f.is_dir() and f.name.startswith('S')]
    
    if not s_folders:
        print("Warning: No folders starting with 'S' found.")
        return

    print(f"Found {len(s_folders)} 'S' folders: {[f.name for f in s_folders]}")

    # 3. Search for CSV files and extract data
    files_found_myo = 0
    files_found_proc = 0 # NEW

    for folder in s_folders:
        for csv_file in folder.rglob("predictions_log.csv"):
            try:
                # Read the CSV file
                df = pd.read_csv(csv_file)
                
                # --- A. Process 'myo_inf_time' (Existing logic) ---
                if "myo_inf_time" in df.columns:
                    all_myo_data_series.append(df["myo_inf_time"])
                    files_found_myo += 1
                else:
                    print(f"  > Warning: Column 'myo_inf_time' not found in {csv_file}")
                
                # --- B. NEW: Process 'timestamp' for processing time ---
                if "timestamp" in df.columns:
                    if len(df) < 2:
                        print(f"  > Info: File {csv_file} has fewer than 2 rows, cannot calculate diff for 'timestamp'.")
                    else:
                        try:
                            # Convert 'timestamp' column to datetime objects
                            # Expected format: HH_MM_SS_MS (e.g., 11_37_55_290)
                            timestamps = pd.to_datetime(df["timestamp"], format="%H_%M_%S_%f", errors='coerce')
                            
                            # Calculate difference between consecutive rows
                            proc_times = timestamps.diff()
                            
                            # Remove the first value (which is NaT/NaN)
                            proc_times = proc_times.dropna()
                            
                            if not proc_times.empty:
                                # Convert Timedelta to milliseconds
                                proc_times_ms = proc_times.dt.total_seconds() * 1000
                                all_proc_time_series.append(proc_times_ms)
                                files_found_proc += 1
                            else:
                                print(f"  > Warning: Invalid or insufficient 'timestamp' data in {csv_file}")
                                
                        except Exception as e:
                            print(f"  > ERROR converting 'timestamp' in {csv_file}: {e}")
                else:
                    print(f"  > Warning: Column 'timestamp' not found in {csv_file}")
            
            except pd.errors.EmptyDataError:
                print(f"  > Warning: Empty file ignored {csv_file}")
            except Exception as e:
                print(f"  > Error reading {csv_file}: {e}")

    # --- 4. Aggregate and Save 'myo_inf_time' ---
    if not all_myo_data_series:
        print("\nOperation terminated: No 'myo_inf_time' data found.")
    else:
        print(f"\nFound and processed {files_found_myo} files for 'myo_inf_time'.")
        
        combined_myo_data = pd.concat(all_myo_data_series, ignore_index=True)
        combined_myo_data.name = "myo_inf_time"

        try:
            combined_myo_data.to_csv(myo_inf_csv_file, index=False, header=True)
            print(f"Aggregated 'myo_inf_time' data saved successfully to:\n{myo_inf_csv_file}")
        except Exception as e:
            print(f"\nError saving CSV file 'myo_inf_time': {e}")
            return

        # Calculate and save statistics for 'myo_inf_time'
        print("Calculating statistics for 'myo_inf_time'...")
        try:
            stats_myo = combined_myo_data.describe()
            
            stats_content = f"""
Statistics for 'myo_inf_time'
-----------------------------------
Source Folder:       {base_path}
Files Processed:     {files_found_myo}
-----------------------------------

Main Metrics:
  Mean:            {stats_myo.loc['mean']:.6f}
  Std Deviation:   {stats_myo.loc['std']:.6f}

Descriptive Metrics:
  Total Samples:   {int(stats_myo.loc['count'])}
  Minimum Value:   {stats_myo.loc['min']:.6f}
  25th Percentile: {stats_myo.loc['25%']:.6f}
  Median (50th):   {stats_myo.loc['50%']:.6f}
  75th Percentile: {stats_myo.loc['75%']:.6f}
  Maximum Value:   {stats_myo.loc['max']:.6f}
"""
            with open(myo_inf_stats_file, "w", encoding="utf-8") as f:
                f.write(stats_content)
            
            print(f"Statistics for 'myo_inf_time' saved successfully to:\n{myo_inf_stats_file}")

        except Exception as e:
            print(f"Error calculating or saving 'myo_inf_time' statistics: {e}")

    # --- 5. NEW: Aggregate and Save 'processing_time_ms' ---
    if not all_proc_time_series:
        print("\nOperation terminated: No 'processing_time' calculated.")
    else:
        print(f"\nFound and processed {files_found_proc} files for 'processing_time'.")
        
        combined_proc_data = pd.concat(all_proc_time_series, ignore_index=True)
        combined_proc_data.name = "processing_time_ms"

        try:
            combined_proc_data.to_csv(proc_time_csv_file, index=False, header=True)
            print(f"Aggregated 'processing_time_ms' data saved successfully to:\n{proc_time_csv_file}")
        except Exception as e:
            print(f"\nError saving CSV file 'processing_time_ms': {e}")
            return

        # Calculate and save statistics for 'processing_time_ms'
        print("Calculating statistics for 'processing_time_ms'...")
        try:
            stats_proc = combined_proc_data.describe()
            
            stats_content = f"""
Statistics for 'processing_time_ms' (Time between consecutive timestamps)
-----------------------------------
Source Folder:       {base_path}
Files Processed:     {files_found_proc}
-----------------------------------

Main Metrics (in milliseconds):
  Mean:            {stats_proc.loc['mean']:.6f} ms
  Std Deviation:   {stats_proc.loc['std']:.6f} ms

Descriptive Metrics (in milliseconds):
  Total Samples:   {int(stats_proc.loc['count'])}
  Minimum Value:   {stats_proc.loc['min']:.6f} ms
  25th Percentile: {stats_proc.loc['25%']:.6f} ms
  Median (50th):   {stats_proc.loc['50%']:.6f} ms
  75th Percentile: {stats_proc.loc['75%']:.6f} ms
  Maximum Value:   {stats_proc.loc['max']:.6f} ms
"""
            with open(proc_time_stats_file, "w", encoding="utf-8") as f:
                f.write(stats_content)
            
            print(f"Statistics for 'processing_time_ms' saved successfully to:\n{proc_time_stats_file}")

        except Exception as e:
            print(f"Error calculating or saving 'processing_time_ms' statistics: {e}")

    print("\n--- Operation Completed ---")


# --- SCRIPT START ---
if __name__ == "__main__":
    # Insert your base path here. 
    # Using r"..." (raw string) prevents issues with Windows backslashes \
    base_repo_path = r"C:\Users\nicol\Thesis\DATA real time\_test_raw"
    
    process_thesis_logs(base_repo_path)

Cartella output 'myo_inf' creata (o già esistente): C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\myo_inf
Cartella output 'proc_time' creata (o già esistente): C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\proc_time

Inizio ricerca in: C:\Users\nicol\Thesis\DATA real time\_test_raw
Trovate 12 cartelle 'S': ['S03', 'S08', 'S10', 'S12', 'S13', 'S15', 'S16', 'S17', 'S18', 'S19', 'S20', 'S21']

Trovati e processati 63 file per 'myo_inf_time'.
Dati 'myo_inf_time' aggregati salvati con successo in:
C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\myo_inf\compiled_myo_inf_time.csv
Calcolo statistiche 'myo_inf_time'...
Statistiche 'myo_inf_time' salvate con successo in:
C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\myo_inf\statistics_myo_inf.txt

Trovati e processati 63 file per 'processing_time'.
Dati 'processing_time_ms' aggregati salvati con successo in:
C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\proc_time\compiled_processing_time_ms.csv
Calcolo stati

In [ ]:
import pandas as pd
from pathlib import Path
import sys

"""
Script to process realtime logs.
It performs the following steps:
1. Searches for 'realtime_log.csv' in all 'S' subfolders of the base path.
2. Checks the 'gate_cv' column.
3. If gate_cv == 1:
   - Extracts 'cv_inf_ms' and 'proc_time_ms'.
   - Saves aggregated data to 'gate_1_data.csv'.
   - Saves statistics for both columns to 'gate_1_statistics.txt'.
4. If gate_cv == 0:
   - Extracts 'proc_time_ms'.
   - Saves aggregated data to 'gate_0_proc_time.csv'.
   - Saves statistics for this column to 'gate_0_statistics.txt'.
5. Saves everything to ../data/all/cv_inf (relative to the script).
"""

def process_realtime_log(base_repo_path):
    
    # 1. Path Definitions
    
    # INPUT Path: Where the data is located
    base_path = Path(base_repo_path)
    
    # OUTPUT Path: Relative to the script's location
    try:
        script_dir = Path(__file__).resolve().parent
        output_dir = script_dir.parent / "data" / "all" / "cv_inf"
    except NameError:
        print("--- ERROR ---")
        print("It seems you are running this code in an interactive environment (e.g., Jupyter).")
        print("In this case, '__file__' is not defined.")
        print("I am using the 'Current Working Directory' (cwd) as the base for output.")
        print("Make sure your .ipynb notebook is in the correct 'scripts' folder.")
        script_dir = Path.cwd().resolve()
        output_dir = script_dir.parent / "data" / "all" / "cv_inf"
        print("----------------")

    # Create the output directory
    try:
        output_dir.mkdir(parents=True, exist_ok=True)
        print(f"Output folder created (or already exists): {output_dir}")
    except Exception as e:
        print(f"Fatal Error: Cannot create output folder {output_dir}.")
        print(f"Details: {e}")
        sys.exit(1)

    # Output files
    gate_1_csv_file = output_dir / "gate_1_data.csv"
    gate_1_stats_file = output_dir / "gate_1_statistics.txt"
    gate_0_csv_file = output_dir / "gate_0_proc_time.csv"
    gate_0_stats_file = output_dir / "gate_0_statistics.txt"

    # Lists to hold extracted data
    gate_1_data_list = []
    gate_0_data_list = []
    
    # Required columns
    required_columns = ['cv_inf_ms', 'proc_time_ms', 'gate_cv']

    print(f"Starting data search in: {base_path}")

    # 2. Find all folders starting with 'S'
    s_folders = [f for f in base_path.iterdir() if f.is_dir() and f.name.startswith('S')]
    
    if not s_folders:
        print("Warning: No folders starting with 'S' found.")
        return

    print(f"Found {len(s_folders)} 'S' folders. Starting scan...")

    # 3. Search for CSV files and process them
    files_found = 0
    for folder in s_folders:
        for csv_file in folder.rglob("realtime_log.csv"):
            try:
                df = pd.read_csv(csv_file)
                files_found += 1

                # Check if required columns exist
                if not all(col in df.columns for col in required_columns):
                    print(f"  > Warning: File ignored {csv_file}")
                    print(f"    Missing one or more columns: {required_columns}")
                    continue

                # --- Case 1: gate_cv == 1 ---
                df_gate_1 = df.loc[df['gate_cv'] == 1, ['cv_inf_ms', 'proc_time_ms']]
                if not df_gate_1.empty:
                    gate_1_data_list.append(df_gate_1)

                # --- Case 2: gate_cv == 0 ---
                df_gate_0 = df.loc[df['gate_cv'] == 0, ['proc_time_ms']]
                if not df_gate_0.empty:
                    gate_0_data_list.append(df_gate_0)

            except pd.errors.EmptyDataError:
                print(f"  > Warning: Empty file ignored {csv_file}")
            except Exception as e:
                print(f"  > Error reading {csv_file}: {e}")

    print(f"\nScan completed. Processed {files_found} 'realtime_log.csv' files.")

    # 4. Process and Save Data for GATE_CV == 1
    if gate_1_data_list:
        print("\n--- Processing data for gate_cv == 1 ---")
        combined_gate_1 = pd.concat(gate_1_data_list, ignore_index=True)
        
        # Save CSV
        try:
            combined_gate_1.to_csv(gate_1_csv_file, index=False)
            print(f"Data 'gate_1' saved successfully to:\n{gate_1_csv_file}")
        except Exception as e:
            print(f"Error saving {gate_1_csv_file}: {e}")

        # Calculate and save statistics
        try:
            stats_content = f"""
Statistics for gate_cv == 1
-----------------------------------
Files 'realtime_log.csv' processed: {files_found}
Total samples (with gate_cv == 1):  {len(combined_gate_1)}
-----------------------------------

Column: 'cv_inf_ms'
  Mean:            {combined_gate_1['cv_inf_ms'].mean():.6f}
  Std Deviation:   {combined_gate_1['cv_inf_ms'].std():.6f}
  Minimum:         {combined_gate_1['cv_inf_ms'].min():.6f}
  Median:          {combined_gate_1['cv_inf_ms'].median():.6f}
  Maximum:         {combined_gate_1['cv_inf_ms'].max():.6f}

Column: 'proc_time_ms'
  Mean:            {combined_gate_1['proc_time_ms'].mean():.6f}
  Std Deviation:   {combined_gate_1['proc_time_ms'].std():.6f}
  Minimum:         {combined_gate_1['proc_time_ms'].min():.6f}
  Median:          {combined_gate_1['proc_time_ms'].median():.6f}
  Maximum:         {combined_gate_1['proc_time_ms'].max():.6f}
"""
            with open(gate_1_stats_file, "w", encoding="utf-8") as f:
                f.write(stats_content)
            print(f"Statistics 'gate_1' saved successfully to:\n{gate_1_stats_file}")

        except Exception as e:
            print(f"Error calculating/saving statistics 'gate_1': {e}")
    else:
        print("\nNo data found for gate_cv == 1.")

    # 5. Process and Save Data for GATE_CV == 0
    if gate_0_data_list:
        print("\n--- Processing data for gate_cv == 0 ---")
        combined_gate_0 = pd.concat(gate_0_data_list, ignore_index=True)
        
        # Save CSV
        try:
            combined_gate_0.to_csv(gate_0_csv_file, index=False)
            print(f"Data 'gate_0' saved successfully to:\n{gate_0_csv_file}")
        except Exception as e:
            print(f"Error saving {gate_0_csv_file}: {e}")

        # Calculate and save statistics
        try:
            stats_content = f"""
Statistics for gate_cv == 0
-----------------------------------
Files 'realtime_log.csv' processed: {files_found}
Total samples (with gate_cv == 0):  {len(combined_gate_0)}
-----------------------------------

Column: 'proc_time_ms'
  Mean:            {combined_gate_0['proc_time_ms'].mean():.6f}
  Std Deviation:   {combined_gate_0['proc_time_ms'].std():.6f}
  Minimum:         {combined_gate_0['proc_time_ms'].min():.6f}
  Median:          {combined_gate_0['proc_time_ms'].median():.6f}
  Maximum:         {combined_gate_0['proc_time_ms'].max():.6f}
"""
            with open(gate_0_stats_file, "w", encoding="utf-8") as f:
                f.write(stats_content)
            print(f"Statistics 'gate_0' saved successfully to:\n{gate_0_stats_file}")

        except Exception as e:
            print(f"Error calculating/saving statistics 'gate_0': {e}")
    else:
        print("\nNo data found for gate_cv == 0.")

    print("\n--- Operation Completed ---")

# --- SCRIPT START ---
if __name__ == "__main__":
    # INPUT Path: where the data to analyze is located.
    base_data_path = r"C:\Users\nicol\Thesis\DATA real time\_test_raw"
    
    process_realtime_log(base_data_path)

--- ERRORE ---
Sembra che tu stia eseguendo questo codice in un ambiente interattivo (es. Jupyter).
In questo caso, '__file__' non è definito.
Sto usando la 'Cartella di Lavoro Corrente' (cwd) come base per l'output.
Assicurati che il tuo notebook .ipynb sia nella cartella 'scripts' corretta.
----------------
Cartella di output creata (o già esistente): C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\cv_inf
Inizio ricerca dati in: C:\Users\nicol\Thesis\DATA real time\_test_raw
Trovate 12 cartelle 'S'. Inizio scansione...

Scansione completata. Processati 49 file 'realtime_log.csv'.

--- Processando dati per gate_cv == 1 ---
Dati 'gate_1' salvati con successo in:
C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\cv_inf\gate_1_data.csv
Statistiche 'gate_1' salvate con successo in:
C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\cv_inf\gate_1_statistics.txt

--- Processando dati per gate_cv == 0 ---
Dati 'gate_0' salvati con successo in:
C:\Users\nicol\Thesis\pyl_est\offli

In [ ]:
import pandas as pd
from pathlib import Path
import sys

"""
Script to process classes logs.
It performs the following steps:
1. Searches for 'classes_log.csv' in all 'S' subfolders of the base path.
2. Checks the 'label' column.
3. If label != 0:
   - Extracts the 'transition_time' column.
4. Saves aggregated data to 'classes_log_transition_time.csv'.
5. Saves statistics to 'classes_log_statistics.txt',
   including a sample count for each subject 'S'.
6. Saves everything to ../data/all/myo_inf (relative to the script).
"""

def process_classes_log(base_repo_path):
    
    # 1. Path Definitions
    
    # INPUT Path: Where the data is located
    base_path = Path(base_repo_path)
    
    # OUTPUT Path: Relative to the script's location
    try:
        script_dir = Path(__file__).resolve().parent
        output_dir = script_dir.parent / "data" / "all" / "myo_inf"
    except NameError:
        print("--- ERROR DETECTED (Jupyter/Interactive) ---")
        print("I am using the 'Current Working Directory' (cwd) as the base for output.")
        print("Make sure your .ipynb notebook is in the correct 'scripts' folder.")
        script_dir = Path.cwd().resolve()
        output_dir = script_dir.parent / "data" / "all" / "myo_inf"
        print("---------------------------------------------")

    # Create the output directory
    try:
        output_dir.mkdir(parents=True, exist_ok=True)
        print(f"Output folder created (or already exists): {output_dir}")
    except Exception as e:
        print(f"Fatal Error: Cannot create output folder {output_dir}.")
        print(f"Details: {e}")
        sys.exit(1)

    # Output files (specific names to avoid overwriting)
    output_csv_file = output_dir / "classes_log_transition_time.csv"
    output_stats_file = output_dir / "classes_log_statistics.txt"

    # List to hold extracted data
    data_list = []
    
    # --- NEW ---
    # Dictionary to track counts per subject
    subject_counts = {}
    
    # Required columns
    required_columns = ['label', 'transition_time']

    print(f"Starting data search in: {base_path}")

    # 2. Find all folders starting with 'S'
    s_folders = [f for f in base_path.iterdir() if f.is_dir() and f.name.startswith('S')]
    
    if not s_folders:
        print("Warning: No folders starting with 'S' found.")
        return

    print(f"Found {len(s_folders)} 'S' folders. Starting scan...")

    # 3. Search for CSV files and process them
    files_found = 0
    for folder in s_folders:
        
        # --- NEW ---
        # Counter for the current subject
        current_subject_count = 0
        
        for csv_file in folder.rglob("classes_log.csv"):
            try:
                df = pd.read_csv(csv_file)
                files_found += 1

                if not all(col in df.columns for col in required_columns):
                    print(f"  > Warning: File ignored {csv_file}")
                    print(f"    Missing one or more columns: {required_columns}")
                    continue

                # --- Apply filter: label != 0 ---
                df_filtered = df.loc[df['label'] != 0, ['transition_time']]
                
                if not df_filtered.empty:
                    data_list.append(df_filtered)
                    
                    # --- NEW ---
                    # Update count for this subject
                    current_subject_count += len(df_filtered)

            except pd.errors.EmptyDataError:
                print(f"  > Warning: Empty file ignored {csv_file}")
            except Exception as e:
                print(f"  > Error reading {csv_file}: {e}")
        
        # --- NEW ---
        # Save total count for the subject, even if it is 0
        subject_counts[folder.name] = current_subject_count
        if current_subject_count > 0:
            print(f"  > Found {current_subject_count} samples for subject {folder.name}")

    print(f"\nScan completed. Processed {files_found} 'classes_log.csv' files.")

    # 4. Process and Save Data
    if data_list:
        print("\n--- Processing data for label != 0 ---")
        combined_data = pd.concat(data_list, ignore_index=True)
        
        # Save CSV
        try:
            combined_data.to_csv(output_csv_file, index=False)
            print(f"Data 'transition_time' (label != 0) saved successfully to:\n{output_csv_file}")
        except Exception as e:
            print(f"Error saving {output_csv_file}: {e}")

        # Calculate and save statistics
        try:
            stats_data = combined_data['transition_time']
            
            # --- NEW ---
            # Build the string for subject summary
            subject_stats_lines = ["\nSample count per subject (where label != 0):"]
            if not subject_counts:
                subject_stats_lines.append("  No subjects analyzed.")
            else:
                # Sort subjects by name (e.g., S01, S02, S10)
                for subject, count in sorted(subject_counts.items()):
                    subject_stats_lines.append(f"  - {subject}: {count} samples")
            
            subject_stats_str = "\n".join(subject_stats_lines)

            # --- MODIFIED ---
            # Added 'subject_stats_str' block to the text file
            stats_content = f"""
Statistics for 'transition_time' (where label != 0)
---------------------------------------------------
Files 'classes_log.csv' processed: {files_found}
Total samples (with label != 0):   {len(stats_data)}
---------------------------------------------------
{subject_stats_str}
---------------------------------------------------

Aggregate Metrics ('transition_time'):
  Mean:            {stats_data.mean():.6f}
  Std Deviation:   {stats_data.std():.6f}
  Minimum:         {stats_data.min():.6f}
  25th Percentile: {stats_data.quantile(0.25):.6f}
  Median (50th):   {stats_data.median():.6f}
  75th Percentile: {stats_data.quantile(0.75):.6f}
  Maximum:         {stats_data.max():.6f}
"""
            with open(output_stats_file, "w", encoding="utf-8") as f:
                f.write(stats_content)
            print(f"Statistics 'transition_time' (with subject counts) saved successfully to:\n{output_stats_file}")

        except Exception as e:
            print(f"Error calculating/saving statistics: {e}")
    else:
        print("\nNo data found with 'label' not equal to 0.")

    print("\n--- Operation Completed ---")

# --- SCRIPT START ---
if __name__ == "__main__":
    # INPUT Path: where the data to analyze is located.
    base_data_path = r"C:\Users\nicol\Thesis\DATA real time\_test_raw"
    
    process_classes_log(base_data_path)

--- ERRORE RILEVATO (Jupyter/Interattivo) ---
Sto usando la 'Cartella di Lavoro Corrente' (cwd) come base per l'output.
Assicurati che il tuo notebook .ipynb sia nella cartella 'scripts' corretta.
---------------------------------------------
Cartella di output creata (o già esistente): C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\myo_inf
Inizio ricerca dati in: C:\Users\nicol\Thesis\DATA real time\_test_raw
Trovate 12 cartelle 'S'. Inizio scansione...
  > Trovati 35 campioni per il soggetto S03
  > Trovati 39 campioni per il soggetto S08
  > Trovati 33 campioni per il soggetto S10
  > Attenzione: File vuoto ignorato C:\Users\nicol\Thesis\DATA real time\_test_raw\S12\test2b\5\classes_log.csv
  > Trovati 36 campioni per il soggetto S12
  > Trovati 36 campioni per il soggetto S13
  > Trovati 37 campioni per il soggetto S15
  > Trovati 30 campioni per il soggetto S16
  > Trovati 36 campioni per il soggetto S17
  > Trovati 31 campioni per il soggetto S18
  > Trovati 31 campioni pe

In [ ]:
import pandas as pd
from pathlib import Path
import sys

"""
Script to process EMG logs.
It performs the following steps:
1. Searches for 'raw_emg_imu.csv' in all 'S' subfolders.
2. For each file, calculates an estimated sampling frequency.
3. Frequency Logic:
   - N_valid = (Total Rows) - (Count of the first timestamp)
   - Delta_T = (Last timestamp) - (First timestamp) [in seconds]
   - Frequency (Hz) = N_valid / Delta_T
4. Saves all frequencies to 'frequencies_emg.csv'.
5. Saves statistics to 'frequencies_emg_statistics.txt'.
6. Saves everything to ../data/all/myo_inf (relative to the script).
"""

def process_emg_log(base_repo_path):
    
    # 1. Path Definitions
    
    # INPUT Path: Where the data is located
    base_path = Path(base_repo_path)
    
    # OUTPUT Path: Relative to the script's location
    try:
        script_dir = Path(__file__).resolve().parent
        output_dir = script_dir.parent / "data" / "all" / "myo_inf"
    except NameError:
        print("--- ERROR DETECTED (Jupyter/Interactive) ---")
        print("I am using the 'Current Working Directory' (cwd) as the base for output.")
        print("Make sure your .ipynb notebook is in the correct 'scripts' folder.")
        script_dir = Path.cwd().resolve()
        output_dir = script_dir.parent / "data" / "all" / "myo_inf"
        print("---------------------------------------------")

    # Create the output directory
    try:
        output_dir.mkdir(parents=True, exist_ok=True)
        print(f"Output folder created (or already exists): {output_dir}")
    except Exception as e:
        print(f"Fatal Error: Cannot create output folder {output_dir}.")
        print(f"Details: {e}")
        sys.exit(1)

    # Output files
    output_csv_file = output_dir / "frequencies_emg.csv"
    output_stats_file = output_dir / "frequencies_emg_statistics.txt"

    # List to hold all calculated frequencies
    all_frequencies = []
    
    print(f"Starting data search in: {base_path}")

    # 2. Find all folders starting with 'S'
    s_folders = [f for f in base_path.iterdir() if f.is_dir() and f.name.startswith('S')]
    
    if not s_folders:
        print("Warning: No folders starting with 'S' found.")
        return

    print(f"Found {len(s_folders)} 'S' folders. Starting scan...")

    # 3. Search for CSV files and process them
    files_found = 0
    for folder in s_folders:
        for csv_file in folder.rglob("raw_emg_imu.csv"):
            try:
                # Read only the 'Timestamps' column as string
                df = pd.read_csv(
                    csv_file, 
                    usecols=['Timestamps'], 
                    dtype={'Timestamps': str},
                    skipinitialspace=True # Removes whitespace
                )
                
                # Remove any 'None' or 'NaN' rows
                df.dropna(inplace=True)
                
                total_rows = len(df)

                # Need at least 2 rows to calculate duration
                if total_rows < 2:
                    print(f"  > Warning: File ignored {csv_file} (less than 2 rows)")
                    continue

                # ----------------------------------------------------
                # Start Calculation Logic
                # ----------------------------------------------------
                
                # 1. Find first and last timestamp (as strings)
                first_ts_value = df['Timestamps'].iloc[0]
                last_ts_value = df['Timestamps'].iloc[-1]
                
                # 2. Count occurrences of the first timestamp
                first_ts_count = (df['Timestamps'] == first_ts_value).sum()
                
                # 3. Calculate N_valid (as requested)
                n_valid = total_rows - first_ts_count
                
                if n_valid <= 0:
                    print(f"  > Warning: File ignored {csv_file} (N_valid = {n_valid})")
                    continue

                # 4. Calculate Delta_T (Duration)
                # Convert strings to datetime objects
                # Format %H_%M_%S_%f handles HH_MM_SS_mmm (e.g. 043 -> 43000 microseconds)
                start_time = pd.to_datetime(first_ts_value, format='%H_%M_%S_%f')
                end_time = pd.to_datetime(last_ts_value, format='%H_%M_%S_%f')
                
                # Handle "midnight crossing" case (e.g., 23:59 -> 00:01)
                if end_time < start_time:
                    end_time += pd.Timedelta(days=1)
                
                # Calculate duration in seconds
                delta_t_seconds = (end_time - start_time).total_seconds()
                
                if delta_t_seconds == 0:
                    print(f"  > Warning: File ignored {csv_file} (Total duration is 0.0s)")
                    continue

                # 5. Calculate Frequency (Hz)
                # Formula: Samples / Seconds
                frequency_hz = n_valid / delta_t_seconds
                # ----------------------------------------------------
                
                all_frequencies.append(frequency_hz)
                files_found += 1
                print(f"  > Processed {folder.name}/{csv_file.name}: {frequency_hz:.2f} Hz")

            except pd.errors.EmptyDataError:
                print(f"  > Warning: Empty file ignored {csv_file}")
            except KeyError:
                print(f"  > Warning: Column 'Timestamps' not found in {csv_file}")
            except ValueError as e:
                print(f"  > Error parsing timestamp in {csv_file}: {e}")
            except Exception as e:
                print(f"  > Unknown error reading {csv_file}: {e}")

    print(f"\nScan completed. Processed {files_found} 'raw_emg_imu.csv' files.")

    # 4. Process and Save Data
    if all_frequencies:
        print("\n--- Processing calculated frequencies ---")
        freq_df = pd.DataFrame(all_frequencies, columns=['calculated_frequency_hz'])
        
        # Save CSV
        try:
            freq_df.to_csv(output_csv_file, index=False)
            print(f"Data 'frequencies_emg.csv' saved successfully to:\n{output_csv_file}")
        except Exception as e:
            print(f"Error saving {output_csv_file}: {e}")

        # Calculate and save statistics
        try:
            stats_content = f"""
Statistics for EMG Sampling Frequency (Hz)
---------------------------------------------------
Files 'raw_emg_imu.csv' processed: {files_found}

Methodology Note (as requested):
- N_valid = (Total Rows) - (Count of the first timestamp)
- Delta_T = (Last timestamp) - (First timestamp) [seconds]
- Frequency = N_valid / Delta_T
---------------------------------------------------

Aggregate Metrics:
  Mean:            {freq_df['calculated_frequency_hz'].mean():.3f} Hz
  Std Deviation:   {freq_df['calculated_frequency_hz'].std():.3f} Hz
  Minimum:         {freq_df['calculated_frequency_hz'].min():.3f} Hz
  25th Percentile: {freq_df['calculated_frequency_hz'].quantile(0.25):.3f} Hz
  Median (50th):   {freq_df['calculated_frequency_hz'].median():.3f} Hz
  75th Percentile: {freq_df['calculated_frequency_hz'].quantile(0.75):.3f} Hz
  Maximum:         {freq_df['calculated_frequency_hz'].max():.3f} Hz
"""
            with open(output_stats_file, "w", encoding="utf-8") as f:
                f.write(stats_content)
            print(f"Statistics 'frequencies_emg_statistics.txt' saved successfully to:\n{output_stats_file}")

        except Exception as e:
            print(f"Error calculating/saving statistics: {e}")
    else:
        print("\nNo frequency was calculated.")

    print("\n--- Operation Completed ---")

# --- SCRIPT START ---
if __name__ == "__main__":
    # INPUT Path: where the data to analyze is located.
    base_data_path = r"C:\Users\nicol\Thesis\DATA real time\_test_raw"
    
    process_emg_log(base_data_path)

--- ERRORE RILEVATO (Jupyter/Interattivo) ---
Sto usando la 'Cartella di Lavoro Corrente' (cwd) come base per l'output.
Assicurati che il tuo notebook .ipynb sia nella cartella 'scripts' corretta.
---------------------------------------------
Cartella di output creata (o già esistente): C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\myo_inf
Inizio ricerca dati in: C:\Users\nicol\Thesis\DATA real time\_test_raw
Trovate 12 cartelle 'S'. Inizio scansione...
  > Processato S03/raw_emg_imu.csv: 165.44 Hz
  > Processato S03/raw_emg_imu.csv: 121.84 Hz
  > Processato S03/raw_emg_imu.csv: 123.05 Hz
  > Processato S03/raw_emg_imu.csv: 95.63 Hz
  > Processato S03/raw_emg_imu.csv: 106.68 Hz
  > Processato S08/raw_emg_imu.csv: 191.29 Hz
  > Processato S08/raw_emg_imu.csv: 114.14 Hz
  > Processato S08/raw_emg_imu.csv: 111.47 Hz
  > Processato S08/raw_emg_imu.csv: 109.02 Hz
  > Processato S08/raw_emg_imu.csv: 128.48 Hz
  > Processato S08/raw_emg_imu.csv: 97.37 Hz
  > Processato S08/raw_emg_imu

In [ ]:
import pandas as pd
from pathlib import Path
import sys

"""
Script to process IMU logs.
It performs the following steps:
1. Searches for 'raw_emg_imu.csv' in all 'S' subfolders.
2. Reads 'Timestamps' and the 6 IMU columns.
3. Filters rows:
   - Removes rows where ALL 6 IMU columns are 0.
   - Removes rows with duplicate IMU values (same 6 values).
4. Calculates frequency on filtered data:
   - N_valid = (Total Filtered Rows) - (Count of the first timestamp)
   - Delta_T = (Last timestamp) - (First timestamp) [in seconds]
   - Frequency (Hz) = N_valid / Delta_T
5. Saves all frequencies to 'frequencies_imu.csv'.
6. Saves statistics to 'frequencies_imu_statistics.txt'.
7. Saves everything to ../data/all/myo_inf (relative to the script).
"""

def process_imu_log(base_repo_path):
    
    # 1. Path Definitions
    
    # INPUT Path: Where the data is located
    base_path = Path(base_repo_path)
    
    # OUTPUT Path: Relative to the script's location
    try:
        script_dir = Path(__file__).resolve().parent
        output_dir = script_dir.parent / "data" / "all" / "myo_inf"
    except NameError:
        print("--- ERROR DETECTED (Jupyter/Interactive) ---")
        print("I am using the 'Current Working Directory' (cwd) as the base for output.")
        print("Make sure your .ipynb notebook is in the correct 'scripts' folder.")
        script_dir = Path.cwd().resolve()
        output_dir = script_dir.parent / "data" / "all" / "myo_inf"
        print("---------------------------------------------")

    # Create the output directory
    try:
        output_dir.mkdir(parents=True, exist_ok=True)
        print(f"Output folder created (or already exists): {output_dir}")
    except Exception as e:
        print(f"Fatal Error: Cannot create output folder {output_dir}.")
        print(f"Details: {e}")
        sys.exit(1)

    # Output files
    output_csv_file = output_dir / "frequencies_imu.csv"
    output_stats_file = output_dir / "frequencies_imu_statistics.txt"

    # Define columns
    ts_col = 'Timestamps' # As per your spec
    sensor_cols = ['ACC_X', 'ACC_Y', 'ACC_Z', 'GYR_X', 'GYR_Y', 'GYR_Z']
    cols_to_read = [ts_col] + sensor_cols

    # List to hold all calculated frequencies
    all_frequencies = []
    
    print(f"Starting data search in: {base_path}")

    # 2. Find all folders starting with 'S'
    s_folders = [f for f in base_path.iterdir() if f.is_dir() and f.name.startswith('S')]
    
    if not s_folders:
        print("Warning: No folders starting with 'S' found.")
        return

    print(f"Found {len(s_folders)} 'S' folders. Starting scan...")

    # 3. Search for CSV files and process them
    files_found = 0
    for folder in s_folders:
        for csv_file in folder.rglob("raw_emg_imu.csv"):
            try:
                # Read only columns of interest
                df = pd.read_csv(
                    csv_file, 
                    usecols=cols_to_read, 
                    dtype={ts_col: str}, # Read timestamp as string
                    skipinitialspace=True
                )
                
                # Remove any 'None' or 'NaN' rows
                df.dropna(inplace=True)
                
                rows_initial = len(df)
                if rows_initial < 2:
                    print(f"  > Warning: File ignored {csv_file} (less than 2 rows)")
                    continue
                
                # ----------------------------------------------------
                # Start FILTERING Logic
                # ----------------------------------------------------
                
                # 1. Zero Filter: Find rows where ALL 6 sensors are 0
                is_all_zero = (df[sensor_cols] == 0).all(axis=1)
                
                # Keep only rows that are NOT all zero
                df_filtered_zeros = df[~is_all_zero]
                
                rows_after_zeros = len(df_filtered_zeros)
                if rows_after_zeros < 2:
                    print(f"  > Warning: File ignored {csv_file} (no valid data after zero filter)")
                    continue

                # 2. Duplicate Filter: Remove duplicates BASED ON SENSORS
                df_filtered_final = df_filtered_zeros.drop_duplicates(subset=sensor_cols, keep='first')
                
                rows_final = len(df_filtered_final)
                if rows_final < 2:
                    print(f"  > Warning: File ignored {csv_file} (no valid data after duplicate filter)")
                    continue
                
                # ----------------------------------------------------
                # Start CALCULATION Logic (on filtered data)
                # ----------------------------------------------------
                
                # 1. Find first and last timestamp (as strings)
                first_ts_value = df_filtered_final[ts_col].iloc[0]
                last_ts_value = df_filtered_final[ts_col].iloc[-1]
                
                # 2. Count occurrences of the first timestamp (ON FILTERED DATA)
                first_ts_count = (df_filtered_final[ts_col] == first_ts_value).sum()
                
                # 3. Calculate N_valid
                n_valid = rows_final - first_ts_count
                
                if n_valid <= 0:
                    print(f"  > Warning: File ignored {csv_file} (N_valid = {n_valid})")
                    continue

                # 4. Calculate Delta_T (Duration)
                start_time = pd.to_datetime(first_ts_value, format='%H_%M_%S_%f')
                end_time = pd.to_datetime(last_ts_value, format='%H_%M_%S_%f')
                
                if end_time < start_time:
                    end_time += pd.Timedelta(days=1)
                
                delta_t_seconds = (end_time - start_time).total_seconds()
                
                if delta_t_seconds == 0:
                    print(f"  > Warning: File ignored {csv_file} (Total duration is 0.0s)")
                    continue

                # 5. Calculate Frequency (Hz)
                frequency_hz = n_valid / delta_t_seconds
                
                all_frequencies.append(frequency_hz)
                files_found += 1
                print(f"  > Processed {folder.name}/{csv_file.name}: {frequency_hz:.2f} Hz "
                      f"(Rows: {rows_initial} -> Zeros: {rows_after_zeros} -> Final: {rows_final})")

            except pd.errors.EmptyDataError:
                print(f"  > Warning: Empty file ignored {csv_file}")
            except KeyError as e:
                print(f"  > Warning: Column not found in {csv_file}. (Error: {e})")
            except ValueError as e:
                print(f"  > Error parsing timestamp in {csv_file}: {e}")
            except Exception as e:
                print(f"  > Unknown error reading {csv_file}: {e}")

    print(f"\nScan completed. Processed {files_found} 'raw_emg_imu.csv' files.")

    # 4. Process and Save Data
    if all_frequencies:
        print("\n--- Processing calculated IMU frequencies ---")
        freq_df = pd.DataFrame(all_frequencies, columns=['calculated_imu_frequency_hz'])
        
        # Save CSV
        try:
            freq_df.to_csv(output_csv_file, index=False)
            print(f"Data 'frequencies_imu.csv' saved successfully to:\n{output_csv_file}")
        except Exception as e:
            print(f"Error saving {output_csv_file}: {e}")

        # Calculate and save statistics
        try:
            stats_content = f"""
Statistics for Unique IMU Data Frequency (Hz)
---------------------------------------------------
Files 'raw_emg_imu.csv' processed: {files_found}

Methodology Note (as requested):
1. Read 'Timestamps' and 6 IMU columns.
2. Removed rows where all 6 IMU sensors are 0.
3. Removed duplicates based ONLY on the 6 IMU values.
4. N_valid = (Total Filtered Rows) - (Count of the first timestamp)
5. Delta_T = (Last timestamp) - (First timestamp) [seconds]
6. Frequency = N_valid / Delta_T
---------------------------------------------------

Aggregate Metrics:
  Mean:            {freq_df['calculated_imu_frequency_hz'].mean():.3f} Hz
  Std Deviation:   {freq_df['calculated_imu_frequency_hz'].std():.3f} Hz
  Minimum:         {freq_df['calculated_imu_frequency_hz'].min():.3f} Hz
  25th Percentile: {freq_df['calculated_imu_frequency_hz'].quantile(0.25):.3f} Hz
  Median (50th):   {freq_df['calculated_imu_frequency_hz'].median():.3f} Hz
  75th Percentile: {freq_df['calculated_imu_frequency_hz'].quantile(0.75):.3f} Hz
  Maximum:         {freq_df['calculated_imu_frequency_hz'].max():.3f} Hz
"""
            with open(output_stats_file, "w", encoding="utf-8") as f:
                f.write(stats_content)
            print(f"Statistics 'frequencies_imu_statistics.txt' saved successfully to:\n{output_stats_file}")

        except Exception as e:
            print(f"Error calculating/saving statistics: {e}")
    else:
        print("\nNo IMU frequency was calculated.")

    print("\n--- Operation Completed ---")

# --- SCRIPT START ---
if __name__ == "__main__":
    # INPUT Path: where the data to analyze is located.
    base_data_path = r"C:\Users\nicol\Thesis\DATA real time\_test_raw"
    
    process_imu_log(base_data_path)

--- ERRORE RILEVATO (Jupyter/Interattivo) ---
Sto usando la 'Cartella di Lavoro Corrente' (cwd) come base per l'output.
Assicurati che il tuo notebook .ipynb sia nella cartella 'scripts' corretta.
---------------------------------------------
Cartella di output creata (o già esistente): C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\myo_inf
Inizio ricerca dati in: C:\Users\nicol\Thesis\DATA real time\_test_raw
Trovate 12 cartelle 'S'. Inizio scansione...
  > Processato S03/raw_emg_imu.csv: 46.04 Hz (Righe: 2807 -> Zeri: 2781 -> Finali: 745)
  > Processato S03/raw_emg_imu.csv: 46.89 Hz (Righe: 14052 -> Zeri: 14047 -> Finali: 5407)
  > Processato S03/raw_emg_imu.csv: 47.83 Hz (Righe: 12864 -> Zeri: 12845 -> Finali: 4997)
  > Processato S03/raw_emg_imu.csv: 48.84 Hz (Righe: 25459 -> Zeri: 25456 -> Finali: 13004)
  > Processato S03/raw_emg_imu.csv: 49.19 Hz (Righe: 29451 -> Zeri: 29450 -> Finali: 13585)
  > Processato S08/raw_emg_imu.csv: 49.77 Hz (Righe: 3219 -> Zeri: 3219 -> Final